In [1]:

import mlflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from mlflow.models import infer_signature

from catboost import CatBoostRegressor
import optuna
import shap
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

/media/sda1/repositories/house-prices/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
data = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# MLflow setup
experiment_name = "house_prices"
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(experiment_name)

# Prepare train/val split
X = data.drop(columns=["SalePrice", "Id"])
y = np.log(data["SalePrice"])
X_test = test.drop(columns=["Id"])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Data Preprocessing & Feature engineering


In [3]:
cat_features = X_train.select_dtypes(include="object").columns.tolist()
num_features = X_train.select_dtypes(include="number").columns.tolist()

print("Categorical features:", cat_features)
print("Numerical features:", num_features)


Categorical features: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
Numerical features: ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageA

In [4]:
ordinal_as_cat = [
    "OverallQual", "OverallCond",
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond",
    "KitchenQual", "HeatingQC",
    "GarageQual", "GarageCond",
    "FireplaceQu"
]

cat_features += ordinal_as_cat

cat_features += [
    "MSSubClass",
    "MoSold"
]


In [5]:
def feature_engineering(df):
    df_new = df.copy()
    
    # Ages
    df_new['HouseAge'] = df_new['YrSold'] - df_new['YearBuilt']
    df_new['RemodAge'] = df_new['YrSold'] - df_new['YearRemodAdd']
    df_new['GarageAge'] = df_new['YrSold'] - df_new['GarageYrBlt'].fillna(df_new['YearBuilt'].median())
    
    # Surfaces
    df_new['TotalSF'] = df_new['GrLivArea'] + df_new['TotalBsmtSF']
    df_new['TotalBath'] = df_new['FullBath'] + 0.5*df_new['HalfBath'] + df_new['BsmtFullBath'] + 0.5*df_new['BsmtHalfBath']
    df_new['PorchSF'] = df_new['OpenPorchSF'] + df_new['EnclosedPorch'] + df_new['3SsnPorch'] + df_new['ScreenPorch']
    df_new['TotalPorchDeck'] = df_new['PorchSF'] + df_new['WoodDeckSF']
    
    # Ratios
    df_new['LotRatio'] = df_new['LotArea'] / (df_new['TotalSF'] + 1)  # +1 pour éviter division par zéro
    
    # Flags
    df_new['FireplaceFlag'] = (df_new['Fireplaces'] > 0).astype(int)
    df_new['HasPool'] = (df_new['PoolArea'] > 0).astype(int)
    df_new['HasGarage'] = (df_new['GarageArea'] > 0).astype(int)
    
    # Score global
    df_new['OverallScore'] = df_new['OverallQual'] * df_new['OverallCond']
    
    for col in cat_features:
        df_new[col] = df_new[col].fillna("Missing")
        

    return df_new

# Transformer pour pipeline
feat_eng = FunctionTransformer(feature_engineering)


## 2. Model Training with Optuna Hyperparameter Tuning

In [6]:

cv = KFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 7),
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "task_type": "CPU",
        "verbose": 0
    }

    model = CatBoostRegressor(**params, loss_function="RMSE", cat_features= cat_features)
    pipeline = Pipeline([
        ("feature_engineering", feat_eng),
        ("model", model)
    ])
    
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    return - np.mean(scores)

study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="optuna_catboost"):
    study.optimize(objective, n_trials=25) 
    
    best_params = study.best_params
    best_rmse = study.best_value
    mlflow.log_metrics({"best_rmse": best_rmse})

    
    best_model = CatBoostRegressor(**best_params, loss_function="RMSE", cat_features= cat_features)
    pipeline = Pipeline([
        ("feature_engineering", feat_eng),        
        ("model", best_model)
    ])
    pipeline.fit(X_train, y_train)

    input_example = X_train.sample(5)
    signature = infer_signature(input_example, pipeline.predict(input_example))

    mlflow.sklearn.log_model(
        pipeline,
        name="optuna_catboost",
        signature=signature,
        input_example=input_example
    )
    
    eval_data = pd.DataFrame(X_val, columns=X.columns)
    eval_data["SalePrice"] = y_val

    model_uri = f"runs:/{mlflow.active_run().info.run_id}/optuna_catboost"
    result = mlflow.models.evaluate(
        model_uri,
        eval_data,
        targets="SalePrice",
        model_type="regressor"
    )

print("Best params:", best_params)
print("Best CV RMSE:", best_rmse)
print("Evaluation metrics:", result.metrics)


[I 2026-01-11 08:52:43,251] A new study created in memory with name: no-name-6b204f02-a131-4c28-b9b7-4ed89eec387b


[I 2026-01-11 08:53:35,770] Trial 0 finished with value: 0.12912263111871372 and parameters: {'learning_rate': 0.013365988163467579, 'depth': 5, 'l2_leaf_reg': 3.508448216302092, 'iterations': 1050}. Best is trial 0 with value: 0.12912263111871372.
[I 2026-01-11 09:02:00,229] Trial 1 finished with value: 0.13441490623471397 and parameters: {'learning_rate': 0.04729573962237682, 'depth': 9, 'l2_leaf_reg': 1.1763435991265896, 'iterations': 1220}. Best is trial 0 with value: 0.12912263111871372.
[I 2026-01-11 09:05:11,853] Trial 2 finished with value: 0.12706616237123902 and parameters: {'learning_rate': 0.019969182868701883, 'depth': 7, 'l2_leaf_reg': 1.6361964805004994, 'iterations': 1502}. Best is trial 2 with value: 0.12706616237123902.
[I 2026-01-11 09:15:31,576] Trial 3 finished with value: 0.13169568745340474 and parameters: {'learning_rate': 0.01180733741574256, 'depth': 9, 'l2_leaf_reg': 2.6285745133484713, 'iterations': 1788}. Best is trial 2 with value: 0.12706616237123902.
[I 

0:	learn: 0.3838759	total: 73ms	remaining: 2m 3s
1:	learn: 0.3775186	total: 100ms	remaining: 1m 24s
2:	learn: 0.3710001	total: 122ms	remaining: 1m 9s
3:	learn: 0.3646156	total: 143ms	remaining: 1m
4:	learn: 0.3585618	total: 166ms	remaining: 56.2s
5:	learn: 0.3526733	total: 197ms	remaining: 55.6s
6:	learn: 0.3465346	total: 223ms	remaining: 54s
7:	learn: 0.3411158	total: 247ms	remaining: 52.3s
8:	learn: 0.3358687	total: 276ms	remaining: 51.8s
9:	learn: 0.3306564	total: 296ms	remaining: 50s
10:	learn: 0.3254694	total: 317ms	remaining: 48.7s
11:	learn: 0.3201733	total: 357ms	remaining: 50.2s
12:	learn: 0.3152038	total: 386ms	remaining: 50.1s
13:	learn: 0.3104061	total: 406ms	remaining: 48.9s
14:	learn: 0.3059566	total: 428ms	remaining: 48s
15:	learn: 0.3014815	total: 452ms	remaining: 47.6s
16:	learn: 0.2971164	total: 471ms	remaining: 46.6s
17:	learn: 0.2927979	total: 494ms	remaining: 46.2s
18:	learn: 0.2888087	total: 517ms	remaining: 45.7s
19:	learn: 0.2849359	total: 540ms	remaining: 45.3s

/media/sda1/repositories/house-prices/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/media/sda1/repositories/house-prices/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

🏃 View run optuna_catboost at: http://localhost:5000/#/experiments/750670933042861254/runs/8896997048ee46d18dab769837f4646f
🧪 View experiment at: http://localhost:5000/#/experiments/750670933042861254
Best params: {'learning_rate': 0.02644549660362047, 'depth': 6, 'l2_leaf_reg': 4.2322451202085976, 'iterations': 1699}
Best CV RMSE: 0.12451062459043093
Evaluation metrics: {'score': np.float64(0.9099983482838496), 'example_count': 292, 'mean_absolute_error': 0.08372470428700203, 'mean_squared_error': 0.01679552487487252, 'root_mean_squared_error': 0.12959754964841164, 'sum_on_target': np.float64(3503.3129158965576), 'mean_on_target': np.float64(11.997646972248484), 'r2_score': 0.9099983482838496, 'max_error': 0.7209407418520897, 'mean_absolute_percentage_error': 0.0070558437499686965}


## 3. Feature Importance Analysis

In [7]:
def get_feature_names(column_transformer):
    feature_names = []

    for name, transformer, columns in column_transformer.transformers_:
        if transformer == 'drop' or transformer == 'passthrough':
            continue

        if hasattr(transformer, 'get_feature_names_out'):
            try:
                names = transformer.get_feature_names_out(columns)
            except:
                names = columns
        else:
            names = columns

        feature_names.extend(names)

    return feature_names

# Get feature names after feature engineering
X_transformed = pipeline.named_steps["feature_engineering"].transform(X_train)
feature_names = X_transformed.columns.tolist()

catboost_model = pipeline.named_steps["model"]
importances = catboost_model.get_feature_importance()

feature_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)



**Note**: Keep only the 12 most important features for the final model

In [8]:
importance_threshold = 2.5
top_features = list(feature_importances[feature_importances["importance"] >= importance_threshold]["feature"])
top_features

['TotalSF',
 'OverallScore',
 'GrLivArea',
 'KitchenQual',
 'OverallQual',
 'GarageCars',
 'FireplaceQu',
 'GarageFinish',
 'TotalBath',
 'YearBuilt']

## 4. Simplified Model with Top Features

In [13]:

from sklearn.compose import ColumnTransformer

feature_selector = ColumnTransformer(
    transformers=[
        ("keep", "passthrough", top_features)
    ],
    remainder="drop"
)


with mlflow.start_run(run_name="final_run"):

    # Get indices of categorical features in top_features
    cat_indices = [i for i, f in enumerate(top_features) if f in cat_features]
    
    final_pipeline = Pipeline([
        ("feature_engineering", feat_eng),
        ("select_features", feature_selector),
        ("model", CatBoostRegressor(
            **best_params,
            loss_function="RMSE",
            cat_features=cat_indices
        ))
    ])
    
    final_pipeline.fit(X_train, y_train)

    # Prédictions
    train_preds = final_pipeline.predict(X_train)
    val_preds = final_pipeline.predict(X_val)

    # RMSE
    train_rmse = root_mean_squared_error(y_train, train_preds)
    val_rmse = root_mean_squared_error(y_val, val_preds)

    # Log dans MLflow
    mlflow.log_metrics({"train_rmse": train_rmse, "val_rmse": val_rmse})

    # Sauvegarde du modèle
    mlflow.sklearn.log_model(final_pipeline, name="final_model")


0:	learn: 0.3836360	total: 15.7ms	remaining: 26.7s
1:	learn: 0.3766651	total: 20.2ms	remaining: 17.2s
2:	learn: 0.3698726	total: 24.3ms	remaining: 13.8s
3:	learn: 0.3636718	total: 39.4ms	remaining: 16.7s
4:	learn: 0.3572732	total: 43.7ms	remaining: 14.8s
5:	learn: 0.3510380	total: 49.9ms	remaining: 14.1s
6:	learn: 0.3453512	total: 52.3ms	remaining: 12.6s
7:	learn: 0.3402492	total: 57.3ms	remaining: 12.1s
8:	learn: 0.3347098	total: 59.5ms	remaining: 11.2s
9:	learn: 0.3290826	total: 61.6ms	remaining: 10.4s
10:	learn: 0.3238976	total: 64.6ms	remaining: 9.92s
11:	learn: 0.3185938	total: 69ms	remaining: 9.7s
12:	learn: 0.3133235	total: 71.4ms	remaining: 9.26s
13:	learn: 0.3089309	total: 76.2ms	remaining: 9.17s
14:	learn: 0.3042975	total: 79ms	remaining: 8.87s
15:	learn: 0.2996032	total: 81.5ms	remaining: 8.57s
16:	learn: 0.2951515	total: 84.3ms	remaining: 8.34s
17:	learn: 0.2906056	total: 89.5ms	remaining: 8.36s
18:	learn: 0.2864524	total: 94.8ms	remaining: 8.39s
19:	learn: 0.2826085	total:

2026/01/11 10:33:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run final_run at: http://localhost:5000/#/experiments/750670933042861254/runs/eeccef3590494e278387032a67e90436
🧪 View experiment at: http://localhost:5000/#/experiments/750670933042861254


In [24]:
model = final_pipeline.named_steps["model"]
importances = model.get_feature_importance()
# Get feature names after feature engineering
X_transformed = final_pipeline.named_steps["feature_engineering"].transform(X_train)
features = X_transformed.columns.tolist()

features

['MSSubClass',
 'MSZoning',
 'LotFrontage',
 'LotArea',
 'Street',
 'Alley',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'MasVnrArea',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinSF1',
 'BsmtFinType2',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'KitchenQual',
 'TotRmsAbvGrd',
 'Functional',
 'Fireplaces',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageCars',
 'GarageArea',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'WoodDeckSF',
 'OpenPorchSF',
 'Enc

## 5. Price Prediction Function and Analysis

In [20]:
def predict_price(form_input, pipeline):
    df = pd.DataFrame([form_input])
    y_pred_log = pipeline.predict(df)[0]
    return np.exp(y_pred_log)

In [21]:
# Analyze impact of OverallQual on price
base_form = {
    "OverallQual": 7,
    "GrLivArea": 1500,
    "1stFlrSF": 1200,
    "Fireplaces": 1,
    "TotalBsmtSF": 900,
    "GarageArea": 400,
    "BsmtFinSF1": 600,
    "GarageCars": 2,
    "LotArea": 8000,
    "YearBuilt": 2005,
    "YearRemodAdd": 2010,
    "OverallCond": 8,
    # Needed by feature engineering and final selector
    "YrSold": 2010,
    "FullBath": 2,
    "HalfBath": 1,
    "BsmtFullBath": 1,
    "BsmtHalfBath": 0,
    "OpenPorchSF": 0,
    "EnclosedPorch": 0,
    "3SsnPorch": 0,
    "ScreenPorch": 0,
    "WoodDeckSF": 0,
    "GarageYrBlt": 2005,
    "KitchenQual": "Gd",
    "GarageFinish": "RFn",
    "FireplaceQu": "Gd"
}

qualities = range(5, 11)
prices = []

for q in qualities:
    form = base_form.copy()
    form["OverallQual"] = q
    price = predict_price(form, final_pipeline)
    prices.append(price)

plt.figure(figsize=(10, 6))
plt.plot(qualities, prices, marker='o', linewidth=2)
plt.xlabel("OverallQual")
plt.ylabel("Prix estimé réel")
plt.title("Effet de OverallQual sur le prix")
plt.grid(True)
plt.show()

KeyError: 'PoolArea'

## 6. Test Predictions and Submission

In [19]:
test_preds_log = final_pipeline.predict(X_test[top_features])
test_preds = np.exp(test_preds_log)

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_preds
})

submission.to_csv("../data/submission.csv", index=False)

print(f"Min: {test_preds.min():.2f}, Max: {test_preds.max():.2f}, Mean: {test_preds.mean():.2f}")

result = mlflow.models.evaluate(
    model_uri,
    eval_data,
    targets="SalePrice",
    model_type="regressor"
)


KeyError: "['TotalSF', 'OverallScore', 'TotalBath'] not in index"

## 7. Data Summary

In [18]:
print("Training data statistics:")
print(data["SalePrice"].describe())
print("\nTest predictions statistics:")
print(pd.Series(test_preds).describe())

Training data statistics:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Test predictions statistics:


NameError: name 'test_preds' is not defined